# Chèn dữ liều vào table(food,interaction) từ drugbank

In [1]:
import pandas as pd
import sqlite3
import re

In [2]:
conn = sqlite3.connect("../data/database/drug-food-interaction.db")

c = conn.cursor()

In [3]:
# insert thêm data vào bảng food
foods = pd.read_csv("../data/raw_data/unique_food_items.csv")

try:
    for food in foods["food_name"]:
        c.execute("""
            INSERT OR IGNORE INTO Foods(food_name)
            VALUES (?)
        """, (food,))
    conn.commit()
except Exception as e:
    print("Lỗi:", e)
    conn.rollback()


In [4]:
#Tạo food_id
foods_df = pd.read_sql(
    """
    SELECT * FROM Foods

""",conn
)

food_map = dict(zip(foods_df["food_name"], foods_df["food_id"]))
print("Mã số của milk là:",food_map["milk"])

Mã số của milk là: 72


## Tạo drug_id

In [5]:
# Đọc file csv 
drugbank_df = pd.read_csv("../data/clean_data/clean_drugbank_data.csv")

# tạo danh sách tên thuốc unique
unique_drugs = drugbank_df["drug_name"].dropna().str.strip().unique() 

for drug in unique_drugs:
    c.execute(
        """
        INSERT OR IGNORE INTO Drugs(drug_name)
        VALUES(?)
        """,(drug,)
    )



In [6]:
drug_count = pd.read_sql(
    "SELECT COUNT(*) as n FROM Drugs",
    conn
)

print(drug_count)

      n
0  1502


In [7]:
# kiểm tra sự trùng lặp
drugs = pd.read_sql(
    """
    SELECT *
    FROM Drugs
    """,
    conn
)

drugs["normalized"] = (
    drugs["drug_name"]
    .str.strip()
    .str.lower()
)

duplicates = drugs[
    drugs.duplicated(
        subset=["normalized"],
        keep=False
    )
]

print(duplicates.sort_values("normalized"))

Empty DataFrame
Columns: [drug_id, drug_name, normalized]
Index: []


In [8]:
conn.commit()

In [9]:
# tạo drug_id mapping từ drug_name
drugs_df = pd.read_sql(
    """
    SELECT drug_id,drug_name
    FROM Drugs
    """,conn
)

drug_map = dict(
    zip(drugs_df["drug_name"], drugs_df["drug_id"])
)
print("mã số của thuốc Warfarin là:",drug_map["warfarin"])

mã số của thuốc Warfarin là: 111


## Chèn dữ liệu vào bảng Interactions

In [10]:
# map trả về id trong drug_map
drugbank_df["drug_id"] = drugbank_df["drug_name"].map(drug_map)

drugbank_df["food_id"] = drugbank_df["food_name"].map(food_map)

In [11]:
print(len(drugbank_df["food_name"].unique()))
print(len(foods_df))

missing_foods = (
    drugbank_df[
        drugbank_df["food_id"].isna()
    ]["food_name"]
    .drop_duplicates()
)

print(missing_foods.tolist())

62
83
['piracetam', nan]


In [12]:
# Xóa 'piracetam' do không phải là food
drugbank_df = drugbank_df.dropna(
    subset=["food_name"]
)
drugbank_df = drugbank_df[
    drugbank_df["food_name"] != "piracetam"
]
drugbank_df["food_id"] = (
    drugbank_df["food_name"]
    .map(food_map)
)

In [13]:
print(drugbank_df["drug_id"].isna().sum())
print(drugbank_df["food_id"].isna().sum())

0
0


In [14]:
interactions = drugbank_df[
    [
        "drug_id",
        "food_id",
        "description",
        "source"
    ]
].copy()

interactions["severity"] = 'Unknown'

interactions = interactions[
    [
        "drug_id",
        "food_id",
        "severity",
        "description",
        "source"
    ]
]
interactions = interactions.drop_duplicates(
    subset=[
        "drug_id",
        "food_id",
        "description"
    ]
)


In [15]:
print(interactions.head())
print(len(interactions))
print(interactions.isna().sum())

   drug_id  food_id severity  \
0      727       55  Unknown   
1      727       56  Unknown   
2      727       57  Unknown   
3      727       58  Unknown   
4      727       40  Unknown   

                                         description  \
0  Avoid herbs and supplements with anticoagulant...   
1  Avoid herbs and supplements with anticoagulant...   
2  Avoid herbs and supplements with anticoagulant...   
3  Avoid herbs and supplements with anticoagulant...   
4  Avoid herbs and supplements with anticoagulant...   

                                              source  
0  Knox C, Wilson M, Klinger CM, et al. DrugBank ...  
1  Knox C, Wilson M, Klinger CM, et al. DrugBank ...  
2  Knox C, Wilson M, Klinger CM, et al. DrugBank ...  
3  Knox C, Wilson M, Klinger CM, et al. DrugBank ...  
4  Knox C, Wilson M, Klinger CM, et al. DrugBank ...  
2809
drug_id        0
food_id        0
severity       0
description    0
source         0
dtype: int64


In [16]:
# insert into table Interactions
for _, row in interactions.iterrows():

    c.execute(
        """
        INSERT OR IGNORE INTO Interactions
        (
            drug_id,
            food_id,
            severity,
            description,
            source
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            row["drug_id"],
            row["food_id"],
            row["severity"],
            row["description"],
            row["source"]
        )
    )

conn.commit()

In [17]:
result = pd.read_sql(
    """
    SELECT COUNT(*) AS total
    FROM Interactions
    """,
    conn
)

print(result)

   total
0   3666


In [38]:
print(
    interactions.duplicated(
        subset=[
            "drug_id",
            "food_id",
            "description"
        ]
    ).sum()
)

0


In [19]:
print(
    pd.read_sql(
        "SELECT COUNT(*) AS total FROM Drugs",
        conn
    )
)

print(
    pd.read_sql(
        "SELECT COUNT(*) AS total FROM Foods",
        conn
    )
)

print(
    pd.read_sql(
        "SELECT COUNT(*) AS total FROM Interactions",
        conn
    )
)

   total
0   1502
   total
0     83
   total
0   3666


In [3]:
# Update severity = 'unknown'

# 2. Định nghĩa tập từ khóa phân loại (Dạng list)
high_keywords = [
    "avoid", "do not take", "contraindicated", "fatal", "life-threatening",
    "severe", "toxic", "toxicity", "serotonin syndrome", "hypertensive crisis",
    "cardiovascular collapse", "respiratory depression", "profound sedation",
    "coma", "death", "cardiotoxicity", "hepatotoxicity", "hepatocellular toxicity",
    "liver injury", "liver damage", "pancreatitis", "rhabdomyolysis",
    "nephrotoxicity", "neurotoxic", "disulfiram effect", "disulfiram-like reaction",
    "disulfiram like reaction", "dose-dumping", "seizures",
    "sudden elevation in blood pressure", "risk of bleeding", "bleeding risk",
    "profound cns depression", "may cause toxicity", "potential to cause toxicity",
    "increase the risk of serotonin syndrome", "increase the risk of bleeding",
    "do not take with or immediately after",
]
low_keywords = [
    "no food interactions are expected", "with or without food",
    "food does not affect", "food does not significantly affect",
    "food does not impact", "food has no effect",
    "food has no clinically significant effect", "food has negligible effects",
    "food had negligible effects", "has negligible effects on", "have negligible",
    "has a negligible effect", "negligible effects on",
    "food is not expected to interfere", "does not affect absorption",
    "does not affect bioavailability", "does not affect pharmacokinetics",
    "does not significantly affect absorption", "does not significantly affect pharmacokinetics",
    "does not significantly alter", "unaffected by food",
    "not to a clinically significant extent", "not to a clinically significant degree",
    "not clinically significant", "not clinically meaningful", "not clinically relevant",
    "no clinically significant", "no clinically meaningful", "clinically insignificant",
    "no significant effect", "no significant interaction", "no meaningful clinical effect",
    "mild", "minimal", "unlikely of clinical relevance",
    "does not result in any clinically meaningful differences",
    "does not have a clinically significant effect",
    "food does not appreciably alter", "pharmacokinetics is unaffected",
]

try:
    # 3. Truy vấn lấy ra interaction_id và description của các dòng đang bị 'unknown'
    # Dùng LOWER(severity) để phòng trường hợp bạn lưu là 'Unknown', 'UNKNOWN' hay 'unknown'
    query_select = "SELECT interaction_id, description FROM Interactions WHERE LOWER(severity) = 'unknown';"
    c.execute(query_select)
    rows_to_update = c.fetchall()
    
    total_rows = len(rows_to_update)
    print(f"Tìm thấy {total_rows} dòng có severity = 'unknown' cần xử lý.")
    
    if total_rows == 0:
        print("Không có dòng nào cần cập nhật!")
    else:
        # 4. Duyệt qua từng dòng dữ liệu và tiến hành phân loại, cập nhật
        updated_count = 0
        for interaction_id, description in rows_to_update:
            desc = str(description).lower()
            
            # Logic kiểm tra từ khóa
            if any(word in desc for word in high_keywords):
                new_severity = 'High'
            elif any(word in desc for word in low_keywords):
                new_severity = 'Low'
            else:
                new_severity = 'Moderate' # Mặc định an toàn lâm sàng nếu không rõ ràng
            
            # Thực hiện câu lệnh UPDATE giá trị mới vào đúng dòng dựa trên interaction_id
            query_update = """
                UPDATE Interactions 
                SET severity = ? 
                WHERE interaction_id = ?;
            """
            c.execute(query_update, (new_severity, interaction_id))
            updated_count += 1
            
        # 5. Lưu (Commit) toàn bộ thay đổi vào database
        conn.commit()
        print(f"Đã cập nhật thành công {updated_count}/{total_rows} dòng từ 'unknown' sang nhãn chính xác!")

except sqlite3.Error as e:
    print(f"Có lỗi xảy ra với Database: {e}")
    conn.rollback() # Hoàn tác nếu gặp lỗi để bảo vệ dữ liệu gốc

finally:
    # Đóng kết nối an toàn
    conn.close()
    print("Đã đóng kết nối Database.")

Tìm thấy 2809 dòng có severity = 'unknown' cần xử lý.
Đã cập nhật thành công 2809/2809 dòng từ 'unknown' sang nhãn chính xác!
Đã đóng kết nối Database.
